# Hana PDF 추적성 확인

처리 규칙은 `traceability.py`에 있습니다. 이 노트북은 ZIP에서 PDF 한 건을 꺼내고, 결과를 눈으로 확인하는 보조 도구입니다.
ZIP과 manifest는 수정하지 않습니다. 페이지 원문이 JSON과 노트북 화면에 나타나므로, 결과를 공유하거나 노트북을 저장하기 전 출력물을 지우세요.

In [ ]:
# Path는 파일·폴더 경로를 다루고, sys는 Python 실행 정보와 import 경로를 다룹니다.
from pathlib import Path
import sys

# 현재 노트북 커널이 실제로 쓰는 Python 버전과 실행 파일을 출력합니다.
print("Python 버전:", sys.version.split()[0])
print("Python 실행 파일:", sys.executable)

# 현재 작업 폴더부터 부모 폴더를 올라가며 저장소 루트를 찾습니다.
DATA = Path.cwd().resolve()
while not ((DATA / "pyproject.toml").is_file() and (DATA / "src/card_benefit").is_dir()):
    if DATA.parent == DATA:  # 더 위로 올라갈 폴더가 없으면 저장소를 못 찾은 것입니다.
        raise RuntimeError("저장소 폴더를 찾지 못했습니다. 저장소 안에서 노트북을 여세요.")
    DATA = DATA.parent  # 현재 위치를 부모 폴더로 바꿔 다시 확인합니다.

# common과 payment_card를 Python이 import할 수 있도록 경로에 추가합니다.
sys.path.insert(0, str(DATA / "src"))
sys.path.insert(0, str(DATA / "src/card_benefit"))
print("저장소:", DATA)

In [ ]:
# 임시 폴더, ZIP 파일 복사, 시간 표시를 위한 표준 Python 도구입니다.
from tempfile import gettempdir
from zipfile import ZipFile
from shutil import copyfileobj
from datetime import datetime

# PyMuPDF는 PDF를 읽고 페이지 이미지를 만듭니다.
import pymupdf

# Image와 display는 노트북 화면에 PDF 이미지와 글자를 보여줍니다.
from IPython.display import Image, display

# PDF 처리와 JSON 저장은 노트북에 복사하지 않고 .py 함수를 사용합니다.
from payment_card.file_pipeline.traceability import create_traceability, write_traceability

In [ ]:
# ZIP 파일의 경로와 ZIP 안에서 꺼낼 PDF의 이름을 정합니다.
ZIP_PATH = DATA / "src/data/hana_all/credit-20261031T142134Z-1-001.zip"
PDF_IN_ZIP = "credit/신용_하나더넥스트멤버스_상품설명서(공식PDF)_15299_20250703.pdf"

# 원본 ZIP은 건드리지 않고, PDF 한 개와 결과를 컴퓨터 임시 폴더에 둡니다.
PILOT_DIR = Path(gettempdir()) / "card-pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)  # 임시 폴더가 없으면 만듭니다.
PDF_PATH = PILOT_DIR / "hana_15299.pdf"
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"ZIP을 찾을 수 없습니다: {ZIP_PATH}")

# ZIP 안의 대상 파일을 열고, 임시 PDF로 복사합니다.
with ZipFile(ZIP_PATH) as archive:
    with archive.open(PDF_IN_ZIP) as source, PDF_PATH.open("wb") as target:
        copyfileobj(source, target)
print(f"임시 PDF 준비 완료: {PDF_PATH}")

In [ ]:
# manifest에 적힌 SHA-256과 페이지 수입니다. 실제 값이 다르면 함수가 오류를 냅니다.
EXPECTED_SHA256 = "4a8995350682476505589ca35cfaec8de67f3cde0dcc0e144cdc59c773afbc84"
EXPECTED_PAGE_COUNT = 2

# .py 함수가 PDF를 읽고 페이지 텍스트와 청크를 만듭니다.
trace = create_traceability(
    PDF_PATH,
    expected_sha256=EXPECTED_SHA256,
    expected_page_count=EXPECTED_PAGE_COUNT,
)

# 매번 새 이름을 사용해 이전 JSON 결과를 덮어쓰지 않습니다.
output_path = PILOT_DIR / f"traceability-{datetime.now():%Y%m%d-%H%M%S}.json"
write_traceability(trace, output_path)
print(f"완료: 페이지 {len(trace['pages'])}개, 청크 {len(trace['chunks'])}개")
print(f"JSON 결과: {output_path}")

In [ ]:
# 페이지별 글자 수, 추출 상태, 청크 생성 여부를 요약해서 봅니다.
for page in trace["pages"]:
    print(
        "페이지", page["page_no"],
        "글자 수", len(page["text"]),
        "추출 상태", page["quality"].get("status"),
        "청크", page["chunk_status"],
        "제외 사유", page["excluded_reason"],
    )

In [ ]:
# page_no를 1 또는 2로 바꾸면 확인할 PDF 페이지를 선택할 수 있습니다.
page_no = 1

# 사람이 세는 페이지는 1부터지만 Python 목록은 0부터라 1을 뺍니다.
with pymupdf.open(PDF_PATH) as document:
    page_image = document[page_no - 1].get_pixmap(dpi=120).tobytes("png")

# 위에는 PDF 이미지, 아래에는 같은 페이지에서 추출한 텍스트를 보여줍니다.
display(Image(data=page_image))
print(trace["pages"][page_no - 1]["text"])

In [ ]:
# 각 청크의 페이지와 위치를 사용해 원문 조각을 다시 찾습니다.
for chunk in trace["chunks"]:
    page_text = trace["pages"][chunk["page_no"] - 1]["text"]
    # 저장 위치는 1부터 시작, Python 문자열 인덱스는 0부터 시작하므로 시작 위치에서 1을 뺍니다.
    # 저장된 끝 위치는 포함하는 번호이며, Python 슬라이스의 끝은 제외되므로 그대로 씁니다.
    original_part = page_text[chunk["start_pos"] - 1:chunk["end_pos"]]
    assert original_part == chunk["text"]  # 다르면 오류를 내서 잘못된 연결을 알립니다.
print("모든 청크 위치가 페이지 원문과 일치합니다.")